<a href="https://colab.research.google.com/github/gmauricio-toledo/tda/blob/main/notebooks/Sesi%C3%B3nPr%C3%A1ctica-3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Homología persistente como método para diagnosticar imágenes médicas</h1>

In [ ]:
!pip install giotto-tda

https://www.kaggle.com/datasets/yusufmurtaza01/chest-xray-pneumonia-balanced-dataset



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yusufmurtaza01/chest-xray-pneumonia-balanced-dataset")

print("Path to dataset files:", path)

Dependiendo de la ruta de folders que muestre el resultado de la celda anterior, descomenta la ruta correcta en la siguiente celda

In [ ]:
import os

base_path = '/root/.cache/kagglehub/datasets/yusufmurtaza01/chest-xray-pneumonia-balanced-dataset/versions/1'
# base_path =  '/kaggle/input/chest-xray-pneumonia-balanced-dataset'

train_path = os.path.join(base_path,'train')
test_path = os.path.join(base_path,'test')
val_path = os.path.join(base_path,'val')

Definimos la función que convertirá las imágenes en arreglos de numpy

In [ ]:
import numpy as np
from PIL import Image

def imagen_a_nparray(path, size=(64, 64)):
    """
    Lee una imagen, la convierte a escala de grises, redimensiona y normaliza.
    Retorna array 2D listo para homología cubical.
    """
    img = Image.open(path).convert('L')  # escala de grises
    img = img.resize(size)
    arr = np.array(img, dtype=np.float32)
    arr = arr / 255.0  # normalizar [0,1]
    arr = 1.0 - arr # Probar comentando y descomentando esta linea
    return arr[np.newaxis, :, :]
    # return arr

In [ ]:
import matplotlib.pyplot as plt

path_normal = os.path.join(train_path,'NORMAL/IM-0001-0001.jpeg')
path_pneumonia = os.path.join(train_path,'PNEUMONIA/person1000_virus_1681.jpeg')

sample_img_normal_array = imagen_a_nparray(path_normal)[0]
sample_img_pneumonia_array = imagen_a_nparray(path_pneumonia)[0]

plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(sample_img_normal_array, cmap='gray')
plt.axis('off')
plt.title('Normal')
plt.subplot(1, 2, 2)
plt.imshow(sample_img_pneumonia_array, cmap='gray')
plt.axis('off')
plt.title('Pneumonia')
plt.show()

🔴 Reflexiona los siguientes puntos

* Explora otras imágenes e identifica qué diferencia suele haber entre ambas clases de imágenes.
* ¿Podría ser la homología persistente una herramienta para encontrar rasgos que permitan distinguir entre ambas imágenes?

Construir el arreglo 3-dimensional con las imágenes de rayos X

**Entrenamiento**

In [ ]:
X_list = []
y = []
label_map = {'NORMAL': 0, 'PNEUMONIA': 1}
total = 0
size = (64, 64)  # Este también es un parámetro a probar

for root, dirs, files in os.walk(train_path):
    label = os.path.basename(root)
    print(f"Entrando a: {root} | label: {label}")
    if label not in label_map:
        continue
    for file in files:
        if file.lower().endswith(('.jpeg', '.jpg')):
            arr = imagen_a_nparray(os.path.join(root, file),size=size)
            X_list.append(arr)
            y.append(label_map[label])
            total += 1

X_train = np.concatenate(X_list, axis=0)
y_train = np.array(y)
print(f"\nFinalizado: {total} imágenes | X shape: {X_train.shape} | y shape: {y_train.shape}")

**Prueba**

In [ ]:
X_list = []
y = []
label_map = {'NORMAL': 0, 'PNEUMONIA': 1}
total = 0

for root, dirs, files in os.walk(test_path):
    label = os.path.basename(root)
    print(f"Entrando a: {root} | label: {label}")
    if label not in label_map:
        continue
    for file in files:
        if file.lower().endswith(('.jpeg', '.jpg')):
            arr = imagen_a_nparray(os.path.join(root, file))
            X_list.append(arr)
            y.append(label_map[label])
            total += 1


X_test = np.concatenate(X_list, axis=0)
y_test = np.array(y)
print(f"\nFinalizado: {total} imágenes | X shape: {X_test.shape} | y shape: {y_test.shape}")

Realiza una tarea similar a la de clasificación a la del dataset MNIST. Recuerda que esto implica la elección de:

* Hiperparámetros del preprocesamiento, en este caso el tamaño de la imágen y la inversión de valores de pixeles
* Dimensiones de la homología persistente
* Método de representación vectorial de los diagramas y sus hiperparámetros respectivos
* Clasificador y sus hiperparámetros

Al final, al igual que con el MNIST, reporta el accuracy de prueba y la matriz de confusión.

Para la clase de hoy, basta con tener un modelo funcionando. Para subir a la asignación, probar varias elecciones para tener el mejor modelo que puedas.